# 05 - Detección de Anomalías

## Pregunta de negocio: *"¿Podemos detectar fallos automáticamente?"*

---

## Contexto y Teoría

La **detección de anomalías** es fundamental para la gestión proactiva de una flota. En lugar de esperar a que ocurra una avería, queremos identificar automáticamente comportamientos inusuales que puedan indicar problemas mecánicos, conducción peligrosa o degradación de componentes.

### Tipos de métodos de detección

#### 1. Métodos estadísticos (univariados)
- **Z-score:** Un punto es anómalo si su valor está a más de N desviaciones estándar de la media. Simple pero asume normalidad.
  - $z = \frac{x - \mu}{\sigma}$, anómalo si $|z| > 3$
- **IQR (Rango Intercuartílico):** Más robusto a outliers. Un punto es anómalo si cae fuera de $[Q1 - 1.5 \cdot IQR, Q3 + 1.5 \cdot IQR]$.
- Ya usamos estos métodos en la Fase 2 para describir los datos. Ahora los aplicamos con un propósito operativo.

#### 2. Métodos basados en modelos (multivariados)
- **Isolation Forest:** Construye árboles de decisión aleatorios. Los puntos anómalos son más fáciles de "aislar" (requieren menos splits). No asume distribución específica.
  - Ventaja: Detecta anomalías que solo se manifiestan al combinar múltiples variables.
  - Ejemplo: Velocidad normal + RPM normal individualmente, pero la combinación es imposible.

#### 3. Detección de drift (cambio gradual)
- **Rolling statistics:** Calculamos media móvil y desviación estándar sobre ventanas temporales.
- Si la media móvil de consumo de un vehículo aumenta progresivamente, indica degradación.
- Más útil que detectar puntos individuales: detecta **tendencias** problemáticas.

### Supervisado vs No supervisado

| Aspecto | Supervisado | No supervisado |
|---------|-------------|----------------|
| Requiere etiquetas | Sí ("fallo" / "normal") | No |
| Precisión | Alta si hay buenos datos | Moderada |
| Uso típico | Fallos conocidos y documentados | Descubrir fallos desconocidos |
| En este notebook | No (no tenemos etiquetas de fallo) | Sí |

En este notebook usamos **detección no supervisada** porque no tenemos etiquetas históricas de fallos. Esto es lo más común en la práctica.

---

## 1. Carga de datos

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

vtype_colors = {
    'electrico': '#2ecc71',
    'gasolina': '#3498db',
    'hibrido': '#9b59b6',
    'deportivo': '#e74c3c'
}

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
print(f"Raíz del proyecto: {project_root}")

In [ ]:
# Cargar telemetría
telemetry_files = sorted(glob.glob(os.path.join(project_root, 'data/raw/telemetry/telemetry_*.csv')))
print(f"Archivos encontrados: {len(telemetry_files)}")

df_list = []
for f in telemetry_files:
    df_list.append(pd.read_csv(f, parse_dates=['timestamp']))
telemetry_full = pd.concat(df_list, ignore_index=True)

# Cargar perfiles de flota
fleet = pd.read_csv(os.path.join(project_root, 'data/raw/fleet_profiles.csv'))
telemetry_full = telemetry_full.merge(fleet[['vehicle_id', 'vehicle_type']], on='vehicle_id', how='left')

print(f"Total registros: {len(telemetry_full):,}")

# Muestreo para rendimiento (si el dataset es muy grande)
MAX_ROWS = 500_000
if len(telemetry_full) > MAX_ROWS:
    telemetry = telemetry_full.sample(n=MAX_ROWS, random_state=42).sort_values('timestamp').reset_index(drop=True)
    print(f"Muestra para análisis: {len(telemetry):,} registros")
else:
    telemetry = telemetry_full.copy()
    print("Usando dataset completo.")

# Variables clave para detección de anomalías
anomaly_features = ['speed_kmh', 'fuel_consumption_rate', 'battery_temp_c', 'motor_rpm', 'acceleration_ms2']
available_features = [f for f in anomaly_features if f in telemetry.columns]
print(f"\nVariables para detección: {available_features}")
telemetry[available_features].describe().round(2)

---

## 2. Anomalías univariadas: Z-score e IQR

Repasamos los métodos de la Fase 2, pero ahora con un enfoque operativo: no solo describimos outliers, sino que los interpretamos como posibles problemas reales.

In [ ]:
def detect_zscore(series, threshold=3):
    """Detecta anomalías usando Z-score."""
    z = (series - series.mean()) / series.std()
    return np.abs(z) > threshold

def detect_iqr(series, factor=1.5):
    """Detecta anomalías usando IQR."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    return (series < lower) | (series > upper)

# Aplicar ambos métodos a cada variable
print("=" * 65)
print(f"{'Variable':<25} {'Z-score (n)':<15} {'IQR (n)':<15} {'Total filas':<12}")
print("=" * 65)

zscore_anomalies = pd.DataFrame(index=telemetry.index)
iqr_anomalies = pd.DataFrame(index=telemetry.index)

for feat in available_features:
    clean = telemetry[feat].dropna()
    z_mask = detect_zscore(telemetry[feat].fillna(telemetry[feat].median()))
    iqr_mask = detect_iqr(telemetry[feat].fillna(telemetry[feat].median()))
    zscore_anomalies[feat] = z_mask
    iqr_anomalies[feat] = iqr_mask
    print(f"{feat:<25} {z_mask.sum():<15,} {iqr_mask.sum():<15,} {len(clean):<12,}")

print("=" * 65)
print(f"\nNota: IQR típicamente detecta más anomalías que Z-score.")
print(f"Z-score es más conservador (solo valores extremos, >3 sigmas).")

In [ ]:
# Visualización: distribución con anomalías marcadas
n_feats = len(available_features)
fig, axes = plt.subplots(1, n_feats, figsize=(4 * n_feats, 5))
if n_feats == 1:
    axes = [axes]

for ax, feat in zip(axes, available_features):
    normal = telemetry[~zscore_anomalies[feat]][feat]
    anomalous = telemetry[zscore_anomalies[feat]][feat]

    ax.hist(normal, bins=50, color='#3498db', alpha=0.7, label='Normal', density=True)
    if len(anomalous) > 0:
        ax.hist(anomalous, bins=20, color='#e74c3c', alpha=0.8, label=f'Anomalías ({len(anomalous):,})', density=True)
    ax.set_title(feat, fontweight='bold', fontsize=11)
    ax.legend(fontsize=8)
    ax.set_xlabel('')

plt.suptitle('Distribución de variables con anomalías Z-score marcadas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 3. Isolation Forest: detección multivariada

Isolation Forest puede detectar anomalías que solo se manifiestan cuando combinamos múltiples variables. Por ejemplo, un vehículo con velocidad baja pero consumo altísimo puede ser normal en cada variable por separado, pero la combinación indica un problema.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Preparar features para Isolation Forest
X_anomaly = telemetry[available_features].dropna()
valid_idx = X_anomaly.index

# Escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_anomaly)

# Ajustar Isolation Forest
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,  # Esperamos ~5% de anomalías
    random_state=42,
    n_jobs=-1
)
iso_predictions = iso_forest.fit_predict(X_scaled)
iso_scores = iso_forest.decision_function(X_scaled)

# -1 = anomalía, 1 = normal
is_anomaly = iso_predictions == -1

print(f"Total de registros analizados: {len(X_anomaly):,}")
print(f"Anomalías detectadas: {is_anomaly.sum():,} ({is_anomaly.mean()*100:.1f}%)")
print(f"Registros normales: {(~is_anomaly).sum():,} ({(~is_anomaly).mean()*100:.1f}%)")

In [ ]:
# Proyección PCA para visualizar en 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Scatter con anomalías marcadas
normal_mask = ~is_anomaly
axes[0].scatter(X_pca[normal_mask, 0], X_pca[normal_mask, 1],
               c='#3498db', alpha=0.1, s=5, label='Normal')
axes[0].scatter(X_pca[is_anomaly, 0], X_pca[is_anomaly, 1],
               c='#e74c3c', alpha=0.5, s=15, label='Anomalía')
axes[0].set_title('Anomalías en espacio PCA', fontsize=13, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
axes[0].legend(markerscale=3)

# Score de anomalía
axes[1].hist(iso_scores[normal_mask], bins=50, color='#3498db', alpha=0.7, label='Normal', density=True)
axes[1].hist(iso_scores[is_anomaly], bins=50, color='#e74c3c', alpha=0.7, label='Anomalía', density=True)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.7, label='Umbral')
axes[1].set_title('Distribución del score de anomalía', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Anomaly Score (menor = más anómalo)')
axes[1].set_ylabel('Densidad')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Comparar registros anómalos vs normales
telemetry_with_anomaly = telemetry.loc[valid_idx].copy()
telemetry_with_anomaly['is_anomaly'] = is_anomaly
telemetry_with_anomaly['anomaly_score'] = iso_scores

print("\n" + "=" * 70)
print("COMPARACIÓN: Registros normales vs anómalos (medias)")
print("=" * 70)

comparison = telemetry_with_anomaly.groupby('is_anomaly')[available_features].mean()
comparison.index = ['Normal', 'Anomalía']
print(comparison.round(2).to_string())

print("\n" + "=" * 70)
print("TOP 10 registros más anómalos:")
print("=" * 70)
top_anomalies = telemetry_with_anomaly.nsmallest(10, 'anomaly_score')
display_cols = ['vehicle_id', 'timestamp'] + available_features + ['anomaly_score']
display_cols = [c for c in display_cols if c in top_anomalies.columns]
print(top_anomalies[display_cols].to_string(index=False))

---

## 4. Rolling statistics para detección de drift

Más allá de detectar puntos anómalos individuales, queremos detectar **tendencias de degradación**: vehículos cuyo consumo aumenta gradualmente con el tiempo. Usamos estadísticas de ventana móvil (rolling) para este propósito.

In [ ]:
# Consumo diario por vehículo
telemetry_full['date'] = telemetry_full['timestamp'].dt.date
daily_vehicle = (
    telemetry_full
    .groupby(['vehicle_id', 'date', 'vehicle_type'])['fuel_consumption_rate']
    .mean()
    .reset_index()
)
daily_vehicle['date'] = pd.to_datetime(daily_vehicle['date'])
daily_vehicle = daily_vehicle.sort_values(['vehicle_id', 'date'])

# Calcular rolling mean y rolling std por vehículo (ventana de 7 días)
window_size = 7
rolling_stats = []

for vid, group in daily_vehicle.groupby('vehicle_id'):
    group = group.sort_values('date').copy()
    group['rolling_mean'] = group['fuel_consumption_rate'].rolling(window=window_size, min_periods=3).mean()
    group['rolling_std'] = group['fuel_consumption_rate'].rolling(window=window_size, min_periods=3).std()
    group['upper_band'] = group['rolling_mean'] + 2 * group['rolling_std']
    group['lower_band'] = group['rolling_mean'] - 2 * group['rolling_std']
    rolling_stats.append(group)

rolling_df = pd.concat(rolling_stats, ignore_index=True)

# Detectar vehículos con tendencia creciente (posible degradación)
vehicle_trends = []
for vid, group in rolling_df.groupby('vehicle_id'):
    clean = group.dropna(subset=['rolling_mean'])
    if len(clean) < 5:
        continue
    # Calcular pendiente de la rolling mean
    x = np.arange(len(clean))
    slope, _ = np.polyfit(x, clean['rolling_mean'].values, 1)
    vtype = clean['vehicle_type'].iloc[0]
    vehicle_trends.append({'vehicle_id': vid, 'slope': slope, 'vehicle_type': vtype,
                          'mean_consumption': clean['fuel_consumption_rate'].mean()})

trends_df = pd.DataFrame(vehicle_trends).sort_values('slope', ascending=False)

print("Vehículos con mayor tendencia creciente de consumo (posible degradación):")
print(trends_df.head(10).to_string(index=False))
print(f"\nVehículos con tendencia decreciente (mejorando):")
print(trends_df.tail(5).to_string(index=False))

In [ ]:
# Visualizar: vehículo normal vs vehículo con degradación
degrading_vehicle = trends_df.iloc[0]['vehicle_id']  # Mayor pendiente positiva
normal_vehicle = trends_df[trends_df['slope'].abs() == trends_df['slope'].abs().min()].iloc[0]['vehicle_id']  # Pendiente más cercana a 0

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

for ax, vid, title_prefix in zip(axes, [normal_vehicle, degrading_vehicle], ['Vehículo estable', 'Vehículo con degradación']):
    vdata = rolling_df[rolling_df['vehicle_id'] == vid].copy()
    vtype = vdata['vehicle_type'].iloc[0] if len(vdata) > 0 else 'desconocido'
    color = vtype_colors.get(vtype, '#34495e')

    ax.plot(vdata['date'], vdata['fuel_consumption_rate'], color=color, alpha=0.4, linewidth=1, label='Consumo diario')
    ax.plot(vdata['date'], vdata['rolling_mean'], color=color, linewidth=2.5, label=f'Media móvil ({window_size}d)')
    ax.fill_between(vdata['date'], vdata['lower_band'], vdata['upper_band'],
                   color=color, alpha=0.1, label='Banda de confianza (2σ)')

    slope_val = trends_df[trends_df['vehicle_id'] == vid]['slope'].values[0]
    ax.set_title(f'{title_prefix}: {vid} ({vtype}) — Pendiente: {slope_val:.6f}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Consumo de combustible')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Fecha')
plt.suptitle('Detección de drift: estable vs degradación', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

## 5. Clasificación de severidad de anomalías

No todas las anomalías son iguales. Un solo valor inusual puede ser ruido, pero si múltiples variables son anómalas simultáneamente, es una señal más fuerte. Definimos niveles de severidad:

- **Normal:** 0 variables anómalas
- **Warning (advertencia):** 1-2 variables anómalas
- **Critical (crítico):** 3+ variables anómalas

In [ ]:
# Contar cuántas variables son anómalas por registro (usando IQR)
anomaly_count = iqr_anomalies[available_features].sum(axis=1)
telemetry['anomaly_count'] = anomaly_count

# Clasificar severidad
def classify_severity(count):
    if count == 0:
        return 'Normal'
    elif count <= 2:
        return 'Warning'
    else:
        return 'Critical'

telemetry['severity'] = telemetry['anomaly_count'].apply(classify_severity)

severity_counts = telemetry['severity'].value_counts()
print("Distribución de severidad:")
for level in ['Normal', 'Warning', 'Critical']:
    count = severity_counts.get(level, 0)
    pct = count / len(telemetry) * 100
    print(f"  {level:<10} {count:>10,} ({pct:>5.1f}%)")

In [ ]:
# Heatmap de anomalías por vehículo
vehicle_anomaly_counts = telemetry.groupby('vehicle_id')['anomaly_count'].agg(['mean', 'sum', 'count']).reset_index()
vehicle_anomaly_counts.columns = ['vehicle_id', 'anomaly_mean', 'anomaly_total', 'total_records']
vehicle_anomaly_counts['anomaly_rate'] = vehicle_anomaly_counts['anomaly_total'] / vehicle_anomaly_counts['total_records']
vehicle_anomaly_counts = vehicle_anomaly_counts.merge(fleet[['vehicle_id', 'vehicle_type']], on='vehicle_id', how='left')
vehicle_anomaly_counts = vehicle_anomaly_counts.sort_values('anomaly_rate', ascending=False)

# Heatmap: severidad por vehículo
severity_by_vehicle = telemetry.groupby(['vehicle_id', 'severity']).size().unstack(fill_value=0)
severity_by_vehicle = severity_by_vehicle.reindex(columns=['Normal', 'Warning', 'Critical'], fill_value=0)
# Normalizar por total de registros del vehículo
severity_pct = severity_by_vehicle.div(severity_by_vehicle.sum(axis=1), axis=0) * 100
severity_pct = severity_pct.sort_values('Critical', ascending=False)

# Mostrar top 20 vehículos con más anomalías críticas
top_vehicles = severity_pct.head(20)

fig, ax = plt.subplots(figsize=(10, max(6, len(top_vehicles) * 0.4)))
sns.heatmap(
    top_vehicles,
    annot=True, fmt='.1f', cmap='YlOrRd',
    cbar_kws={'label': '% de registros'},
    ax=ax
)
ax.set_title('Distribución de severidad por vehículo (Top 20 con más anomalías)', fontsize=13, fontweight='bold')
ax.set_xlabel('Nivel de severidad')
ax.set_ylabel('Vehículo')
plt.tight_layout()
plt.show()

---

## 6. Patrones temporales de anomalías

¿Cuándo ocurren más anomalías? ¿Hay ciertos horarios o días de la semana más problemáticos? ¿El tipo de carretera influye?

In [ ]:
# Extraer componentes temporales
telemetry['hour'] = telemetry['timestamp'].dt.hour
telemetry['day_of_week'] = telemetry['timestamp'].dt.dayofweek
day_names = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
telemetry['day_name'] = telemetry['day_of_week'].map(dict(enumerate(day_names)))

# Tasa de anomalías por hora y día de la semana
anomaly_binary = (telemetry['anomaly_count'] > 0).astype(int)
temporal_anomalies = telemetry.groupby(['day_of_week', 'hour']).apply(
    lambda x: (x['anomaly_count'] > 0).mean() * 100
).unstack(level=0)
temporal_anomalies.columns = day_names

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap hora x día
sns.heatmap(
    temporal_anomalies,
    cmap='YlOrRd', annot=True, fmt='.1f',
    cbar_kws={'label': 'Tasa de anomalías (%)'},
    ax=axes[0]
)
axes[0].set_title('Tasa de anomalías por hora y día de la semana', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Día de la semana')
axes[0].set_ylabel('Hora del día')

# Por tipo de carretera (si la columna existe)
if 'road_type' in telemetry.columns:
    road_anomalies = telemetry.groupby('road_type').apply(
        lambda x: pd.Series({
            'tasa_anomalias': (x['anomaly_count'] > 0).mean() * 100,
            'anomalias_criticas': (x['severity'] == 'Critical').mean() * 100,
            'n_registros': len(x)
        })
    ).sort_values('tasa_anomalias', ascending=True)

    colors_road = ['#3498db' if v < road_anomalies['tasa_anomalias'].median() else '#e74c3c'
                   for v in road_anomalies['tasa_anomalias']]
    axes[1].barh(road_anomalies.index, road_anomalies['tasa_anomalias'], color=colors_road, edgecolor='white')
    axes[1].set_title('Tasa de anomalías por tipo de carretera', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Tasa de anomalías (%)')
    axes[1].set_ylabel('Tipo de carretera')
    for i, (idx, row) in enumerate(road_anomalies.iterrows()):
        axes[1].text(row['tasa_anomalias'] + 0.2, i, f"{row['tasa_anomalias']:.1f}%", va='center', fontsize=10)
else:
    # Alternativa: anomalías por tipo de vehículo
    vtype_anomalies = telemetry.groupby('vehicle_type').apply(
        lambda x: (x['anomaly_count'] > 0).mean() * 100
    ).sort_values(ascending=True)
    colors_vt = [vtype_colors.get(vt, '#34495e') for vt in vtype_anomalies.index]
    axes[1].barh(vtype_anomalies.index, vtype_anomalies.values, color=colors_vt, edgecolor='white')
    axes[1].set_title('Tasa de anomalías por tipo de vehículo', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Tasa de anomalías (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Evolución temporal de anomalías
telemetry['date'] = pd.to_datetime(telemetry['timestamp'].dt.date)
daily_anomaly_rate = telemetry.groupby('date').apply(
    lambda x: pd.Series({
        'tasa_warning': (x['severity'] == 'Warning').mean() * 100,
        'tasa_critical': (x['severity'] == 'Critical').mean() * 100,
        'tasa_total': (x['anomaly_count'] > 0).mean() * 100
    })
)

fig, ax = plt.subplots(figsize=(16, 6))
ax.fill_between(daily_anomaly_rate.index, 0, daily_anomaly_rate['tasa_warning'],
               color='#f39c12', alpha=0.5, label='Warning')
ax.fill_between(daily_anomaly_rate.index, daily_anomaly_rate['tasa_warning'],
               daily_anomaly_rate['tasa_warning'] + daily_anomaly_rate['tasa_critical'],
               color='#e74c3c', alpha=0.5, label='Critical')
ax.plot(daily_anomaly_rate.index, daily_anomaly_rate['tasa_total'],
       color='#2c3e50', linewidth=2, label='Total anomalías')

ax.set_title('Evolución diaria de la tasa de anomalías', fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Tasa de anomalías (%)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

---

## 7. Resumen y conclusiones

### Tipos de anomalías detectadas

1. **Anomalías puntuales (univariadas):** Valores extremos en variables individuales como velocidad, consumo o temperatura de batería. Detectadas con Z-score e IQR.

2. **Anomalías multivariadas:** Combinaciones inusuales de variables que no son evidentes individualmente. Detectadas con Isolation Forest. Estas son las más interesantes operativamente.

3. **Drift/Degradación:** Tendencias graduales de deterioro en el rendimiento de vehículos individuales. Detectadas con rolling statistics y análisis de pendientes.

### Estrategia de detección recomendada

Para un sistema de alertas en producción, recomendamos un enfoque de **tres capas:**

| Capa | Método | Propósito | Frecuencia |
|------|--------|-----------|------------|
| 1. Tiempo real | Z-score / IQR | Alertas inmediatas de valores extremos | Cada registro |
| 2. Diaria | Isolation Forest | Detección de patrones multivariados anómalos | Batch diario |
| 3. Semanal | Rolling statistics | Detección de degradación gradual | Análisis semanal |

### Hallazgos clave

- La clasificación de severidad (Normal/Warning/Critical) permite priorizar la atención de la flota.
- Los patrones temporales revelan cuándo y dónde se concentran los problemas.
- Los vehículos con tendencia creciente de consumo son candidatos prioritarios para mantenimiento.

### Próximo notebook

En el [06 - Mantenimiento Predictivo](06_predictive_maintenance.ipynb) combinamos todo lo aprendido para construir un sistema que prediga cuándo un vehículo necesitará mantenimiento, transformando la detección de anomalías en decisiones de negocio concretas.